# SQL-каталог из pandas DataFrame

Ноутбук принимает подготовленные pandas DataFrame и строит каталог результатов SQLGlot-анализа. SQL только парсится и не исполняется; подключения к базе данных нет.

In [ ]:
QUERY_DF_NAME = "my_queries_df"
SCHEMA_DF_NAME = "my_schema_df"  # Use None when no schema DataFrame is available.
DEFAULT_SCHEMA = "public"
OUTPUT_DIR = None
BUILD_HTML = False
AUTO_INSTALL = True
ANALYZER_ARCHIVE_URL = (
    "https://github.com/xtreezzz/greenplum-sql-literal-analyzer/"
    "archive/refs/heads/main.zip"
)

## 1. Зависимости

Импорт выполняется в текущем Python-ядре. Если пакет анализатора отсутствует и `AUTO_INSTALL=True`, ноутбук установит его из публичного архива GitHub вместе с поддерживаемой версией pandas.

In [ ]:
import importlib
import importlib.metadata
import shlex
import subprocess
import sys


def pandas_version_is_supported(installed_version):
    if not isinstance(installed_version, str):
        return False
    major_text = installed_version.partition(".")[0]
    return major_text.isdecimal() and int(major_text) == 2


def pandas_distribution_status():
    try:
        installed_version = importlib.metadata.version("pandas")
    except importlib.metadata.PackageNotFoundError:
        return None, False
    return installed_version, pandas_version_is_supported(installed_version)


def pandas_module_status(pandas_module):
    loaded_version = getattr(pandas_module, "__version__", None)
    return loaded_version, pandas_version_is_supported(loaded_version)


def pandas_version_label(installed_version):
    if installed_version is None:
        return "not found"
    return repr(installed_version)


def import_analyzer():
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"gp-sql-analyzer @ {ANALYZER_ARCHIVE_URL}",
        "pandas>=2,<3",
    ]
    manual_install_command = shlex.join(install_command)
    pandas_was_loaded = "pandas" in sys.modules
    pandas_module = sys.modules.get("pandas") if pandas_was_loaded else None
    restart_required = False

    if pandas_was_loaded:
        initial_version, initial_version_supported = pandas_module_status(
            pandas_module
        )
        if initial_version_supported:
            try:
                from gp_sql_analyzer.dataframe import analyze_dataframe as analyzer
            except ImportError as import_error:
                dependency_error = import_error
            else:
                return pandas_module, analyzer
        else:
            restart_required = True
            dependency_error = ImportError(
                "Loaded pandas module has a missing, unparseable, or "
                f"incompatible version: {pandas_version_label(initial_version)}"
            )
    else:
        initial_version, initial_version_supported = pandas_distribution_status()
        if initial_version_supported:
            try:
                import pandas as pandas_module
            except ImportError as import_error:
                dependency_error = import_error
            else:
                imported_version, imported_version_supported = pandas_module_status(
                    pandas_module
                )
                if not imported_version_supported:
                    initial_version = imported_version
                    restart_required = True
                    dependency_error = ImportError(
                        "Imported pandas module has a missing, unparseable, or "
                        "incompatible version: "
                        f"{pandas_version_label(imported_version)}"
                    )
                else:
                    try:
                        from gp_sql_analyzer.dataframe import (
                            analyze_dataframe as analyzer,
                        )
                    except ImportError as import_error:
                        dependency_error = import_error
                    else:
                        return pandas_module, analyzer
        else:
            dependency_error = ImportError(
                "pandas distribution is missing, unparseable, or incompatible: "
                f"{pandas_version_label(initial_version)}"
            )

    loaded_restart_instruction = ""
    if restart_required:
        loaded_restart_instruction = (
            " Restart the kernel, then recreate or reload the input DataFrames "
            "before rerunning this cell."
        )

    if AUTO_INSTALL is not True:
        raise RuntimeError(
            "Required notebook dependencies (gp-sql-analyzer and pandas>=2,<3) "
            "are unavailable, broken, or incompatible. Found pandas version: "
            f"{pandas_version_label(initial_version)}. Set AUTO_INSTALL=True "
            "or install manually with: "
            f"{manual_install_command}.{loaded_restart_instruction}"
        ) from dependency_error

    try:
        subprocess.check_call(install_command)
    except subprocess.CalledProcessError as install_error:
        raise RuntimeError(
            "Could not install notebook dependencies (gp-sql-analyzer and "
            "pandas>=2,<3) in the current kernel. Run manually: "
            f"{manual_install_command}.{loaded_restart_instruction}"
        ) from install_error

    importlib.invalidate_caches()

    if restart_required:
        installed_version, _ = pandas_distribution_status()
        raise RuntimeError(
            "An incompatible pandas module is already loaded in this kernel "
            f"(found {pandas_version_label(initial_version)}; version on disk "
            f"after installation: {pandas_version_label(installed_version)}). "
            "Restart the kernel, then recreate or reload the input DataFrames "
            "before rerunning this cell. Manual install command: "
            f"{manual_install_command}"
        ) from dependency_error

    if pandas_was_loaded:
        try:
            from gp_sql_analyzer.dataframe import analyze_dataframe as analyzer
        except ImportError as retry_error:
            raise RuntimeError(
                "Notebook dependencies are still unavailable or broken after "
                "installation. Restart the kernel or run manually: "
                f"{manual_install_command}"
            ) from retry_error
        return pandas_module, analyzer

    installed_version, installed_version_supported = pandas_distribution_status()
    if not installed_version_supported:
        raise RuntimeError(
            "pandas>=2,<3 is still unavailable or incompatible after installation "
            f"(found {pandas_version_label(installed_version)}). Run manually: "
            f"{manual_install_command}, then rerun this cell."
        ) from dependency_error

    try:
        import pandas as pandas_module
    except ImportError as retry_error:
        raise RuntimeError(
            "Notebook dependencies are still unavailable or broken after "
            "installation. Restart the kernel or run manually: "
            f"{manual_install_command}"
        ) from retry_error

    imported_version, imported_version_supported = pandas_module_status(
        pandas_module
    )
    if not imported_version_supported:
        raise RuntimeError(
            "The imported pandas module is incompatible after installation "
            f"(found {pandas_version_label(imported_version)}). Restart the "
            "kernel, then recreate or reload the input DataFrames before "
            f"rerunning this cell. Manual command: {manual_install_command}"
        ) from dependency_error

    try:
        from gp_sql_analyzer.dataframe import analyze_dataframe as analyzer
    except ImportError as retry_error:
        raise RuntimeError(
            "Notebook dependencies are still unavailable or broken after "
            "installation. Restart the kernel or run manually: "
            f"{manual_install_command}"
        ) from retry_error

    return pandas_module, analyzer


pd, analyze_dataframe = import_analyzer()
from IPython.display import display

## 2. Загрузка исходных DataFrame

До продолжения создайте или загрузите DataFrame с именем из `QUERY_DF_NAME` и, если `SCHEMA_DF_NAME` не равно `None`, DataFrame схемы с указанным именем. Обязательные колонки запросов: `query_text`, `query_text_template`; схемы: `table_schema`, `table_name`, `column_name`.

In [ ]:
# Create or load my_queries_df and optional my_schema_df above this cell before continuing.

In [ ]:
def resolve_dataframe(variable_name, *, required_columns):
    if not isinstance(variable_name, str) or not variable_name.strip():
        raise ValueError("DataFrame variable name must be a non-empty string.")

    try:
        dataframe = globals()[variable_name]
    except KeyError as missing_variable:
        raise NameError(
            f"Required DataFrame variable {variable_name!r} is not defined."
        ) from missing_variable

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{variable_name!r} must be a pandas DataFrame, "
            f"got {type(dataframe).__name__}."
        )

    missing_columns = [
        column for column in required_columns if column not in dataframe.columns
    ]
    if missing_columns:
        raise ValueError(
            f"DataFrame {variable_name!r} is missing required columns: "
            f"{', '.join(missing_columns)}"
        )
    return dataframe


input_queries_df = resolve_dataframe(
    QUERY_DF_NAME,
    required_columns=("query_text", "query_text_template"),
)
if SCHEMA_DF_NAME is None:
    input_schema_df = None
else:
    input_schema_df = resolve_dataframe(
        SCHEMA_DF_NAME,
        required_columns=("table_schema", "table_name", "column_name"),
    )

# Backward-friendly display name; this is only another reference, not a copy or mutation.
schema_df = input_schema_df

## 3. SQLGlot AST-анализ

Анализатор читает исходные DataFrame без их изменения и возвращает шесть отдельных таблиц результатов.

In [ ]:
result = analyze_dataframe(
    input_queries_df,
    schema_df=input_schema_df,
    default_schema=DEFAULT_SCHEMA,
    output_dir=OUTPUT_DIR,
    build_html=BUILD_HTML,
    example_limit=20,
    source_label=QUERY_DF_NAME,
)
row_analysis_df = result.row_analysis_df
details_df = result.details_df
aggregate_df = result.aggregate_df
catalog_tables_df = result.catalog_tables_df
catalog_columns_df = result.catalog_columns_df
errors_df = result.errors_df
assert len(row_analysis_df) == len(input_queries_df)

## 4. Результаты и артефакты

Ниже отображаются все шесть результирующих DataFrame и пути созданных артефактов. `OUTPUT_DIR=None` ничего не записывает на диск. Для `BUILD_HTML=True` необходимо задать `OUTPUT_DIR`.

In [ ]:
display(row_analysis_df)
display(details_df)
display(aggregate_df)
display(catalog_tables_df)
display(catalog_columns_df)
display(errors_df)

for artifact_name, artifact_path in result.artifact_paths.items():
    print(f"{artifact_name}: {artifact_path}")